# Microbatch Analysis

Analyzes the effect of microbatching on SVD optimizer performance across all datasets.
Microbatching averages groups of per-sample losses before computing the Jacobian,
reducing its row dimension from B to B/mb. At mb=B, the Jacobian is 1×P (standard gradient).

## 1. Setup & Data Loading

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import seaborn as sns
from pathlib import Path
from glob import glob
import warnings
warnings.filterwarnings('ignore')

from style import set_style, lr_labels
set_style()

PLOT_DIR = Path('plots/microbatch')
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {PLOT_DIR.resolve()}")

In [ ]:
# Load results from per-run JSONL files in experiment_results/{scan_name}/
from style import load_results

# Microbatch scan configs
SCAN_CONFIGS = {
    'CIFAR-10 (CE)':       'cifar10_resnet_microbatch_scan',
    'CIFAR-10 (LabelReg)': 'cifar10_resnet_microbatch_labelreg_scan',
    'MNIST (CE)':          'mnist_microbatch_ce_scan',
    'MNIST (LabelReg)':    'mnist_microbatch_labelreg_scan',
    'MNIST (CE, old)':     'mnist_microbatch_scan',
    'Toy 1D':              'toy_1d_microbatch_scan',
    'Polynomial':          'polynomial_microbatch_scan',
}

datasets = {}
for name, scan_name in SCAN_CONFIGS.items():
    try:
        df = load_results(scan_name)
        datasets[name] = df
    except FileNotFoundError:
        print(f"  [skip] {name}: not found")

print(f"\nLoaded {len(datasets)} datasets: {list(datasets.keys())}")


In [ ]:
# Helper functions
def get_loss_curve(row, loss_type='val'):
    return np.array(row['losses'][loss_type])

def get_acc_curve(row, acc_type='val_acc'):
    return np.array(row['losses'].get(acc_type, []))

def sliding_average(data, window=10):
    return np.convolve(data, np.ones(window)/window, mode='valid')

def add_derived_columns(df):
    """Add standard derived columns to a scan DataFrame."""
    df = df.copy()
    df['final_val_loss'] = df.apply(lambda r: r['losses']['val'][-1], axis=1)
    df['final_train_loss'] = df.apply(lambda r: r['losses']['train'][-1], axis=1)
    df['final_val_acc'] = df.apply(lambda r: r['losses'].get('val_acc', [np.nan])[-1], axis=1)
    df['final_train_acc'] = df.apply(lambda r: r['losses'].get('train_acc', [np.nan])[-1], axis=1)
    df['total_time'] = df['losses'].apply(lambda l: l.get('total_time', np.nan))
    df['avg_batch_time_train'] = df['losses'].apply(lambda l: l.get('avg_batch_time_train', np.nan))
    # Effective batch size for SVD = batch_size / microbatch_size
    df['microbatch_size'] = df['microbatch_size'].fillna(1).astype(int)
    df['effective_bs'] = df['batch_size'] // df['microbatch_size']
    return df

for name in datasets:
    datasets[name] = add_derived_columns(datasets[name])

# Print summary
for name, df in datasets.items():
    mb_vals = sorted(df['microbatch_size'].unique())
    print(f"{name}: {len(df)} runs, microbatch_sizes={mb_vals}, batch_size={df['batch_size'].unique()}")

## 2. Best Final Loss vs Microbatch Size

For each microbatch size, find the best SVD config (across lr, k, rtol) and plot the best achievable performance.

In [ ]:
colors = sns.color_palette('deep')
has_acc = any(not np.isnan(df['final_val_acc'].iloc[0]) for df in datasets.values())
ncols = 2 if has_acc else 1

fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 6))
if ncols == 1:
    axes = [axes]

for i, (name, df) in enumerate(datasets.items()):
    bs = df['batch_size'].iloc[0]
    mb_vals = sorted(df['microbatch_size'].unique())
    best_losses = []
    best_accs = []
    for mb in mb_vals:
        mb_df = df[df['microbatch_size'] == mb]
        best_losses.append(mb_df['final_val_loss'].min())
        best_accs.append(mb_df['final_val_acc'].max())

    # Effective rows = bs / mb
    eff_rows = [bs // mb for mb in mb_vals]

    ax = axes[0]
    ax.plot(mb_vals, best_losses, 'o-', color=colors[i % len(colors)], lw=2.5,
            markersize=7, label=name)

    if ncols > 1:
        ax = axes[1]
        valid_accs = [a for a in best_accs if not np.isnan(a)]
        if valid_accs:
            ax.plot(mb_vals[:len(valid_accs)], [a * 100 for a in valid_accs],
                    'o-', color=colors[i % len(colors)], lw=2.5, markersize=7, label=name)

ax = axes[0]
ax.set_xlabel('Microbatch Size')
ax.set_ylabel('Best Final Validation Loss')
ax.set_title('Best SVD Performance vs Microbatch Size')
ax.set_yscale('log')
ax.set_xscale('log', base=2)
ax.legend(frameon=False, fontsize=10)

if ncols > 1:
    ax = axes[1]
    ax.set_xlabel('Microbatch Size')
    ax.set_ylabel('Best Final Validation Accuracy (%)')
    ax.set_title('Best SVD Accuracy vs Microbatch Size')
    ax.set_xscale('log', base=2)
    ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'best_loss_vs_microbatch.pdf')
plt.show()

## 3. Training Curves by Microbatch Size

For each dataset, compare the best SVD training curve at each microbatch size.

In [ ]:
n_datasets = len(datasets)
fig, axes = plt.subplots(1, n_datasets, figsize=(6 * n_datasets, 5), squeeze=False)
axes = axes[0]

cmap = plt.cm.viridis

for col, (name, df) in enumerate(datasets.items()):
    ax = axes[col]
    bs = df['batch_size'].iloc[0]
    mb_vals = sorted(df['microbatch_size'].unique())
    n_mb = len(mb_vals)
    mb_colors = [cmap(j / max(n_mb - 1, 1)) for j in range(n_mb)]

    for j, mb in enumerate(mb_vals):
        mb_df = df[df['microbatch_size'] == mb]
        best_idx = mb_df['final_val_loss'].idxmin()
        row = df.loc[best_idx]
        val_curve = get_loss_curve(row, 'val')
        eff = bs // mb
        ax.plot(range(len(val_curve)), val_curve, color=mb_colors[j], lw=2,
                label=f'mb={mb} (eff={eff})')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation Loss')
    ax.set_yscale('log')
    ax.set_title(name)
    ax.legend(fontsize=8, frameon=False, loc='upper right')

plt.suptitle('Best SVD Training Curves by Microbatch Size', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'training_curves_by_microbatch.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Same but for accuracy (classification datasets only)
class_datasets = {k: v for k, v in datasets.items()
                  if not np.isnan(v['final_val_acc'].iloc[0])}

if class_datasets:
    n_cd = len(class_datasets)
    fig, axes = plt.subplots(1, n_cd, figsize=(6 * n_cd, 5), squeeze=False)
    axes = axes[0]

    for col, (name, df) in enumerate(class_datasets.items()):
        ax = axes[col]
        bs = df['batch_size'].iloc[0]
        mb_vals = sorted(df['microbatch_size'].unique())
        n_mb = len(mb_vals)
        mb_colors = [cmap(j / max(n_mb - 1, 1)) for j in range(n_mb)]

        for j, mb in enumerate(mb_vals):
            mb_df = df[df['microbatch_size'] == mb]
            best_idx = mb_df['final_val_acc'].idxmax()
            row = df.loc[best_idx]
            val_acc = get_acc_curve(row, 'val_acc')
            if len(val_acc) > 0:
                eff = bs // mb
                ax.plot(range(1, len(val_acc)+1), val_acc * 100,
                        color=mb_colors[j], lw=2, label=f'mb={mb} (eff={eff})')

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation Accuracy (%)')
        ax.set_title(name)
        ax.legend(fontsize=8, frameon=False, loc='lower right')

    plt.suptitle('Best SVD Accuracy Curves by Microbatch Size', y=1.02)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'accuracy_curves_by_microbatch.pdf', bbox_inches='tight')
    plt.show()

## 4. Wall Time vs Microbatch Size

How does microbatching affect training speed? The Jacobian is smaller, so each step should be faster,
but the update may be less effective (more epochs needed).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, (name, df) in enumerate(datasets.items()):
    mb_vals = sorted(df['microbatch_size'].unique())
    # Best run per microbatch
    best_times = []
    best_batch_times = []
    for mb in mb_vals:
        mb_df = df[df['microbatch_size'] == mb]
        best_idx = mb_df['final_val_loss'].idxmin()
        best_times.append(df.loc[best_idx, 'total_time'])
        best_batch_times.append(df.loc[best_idx, 'avg_batch_time_train'])

    c = colors[i % len(colors)]
    axes[0].plot(mb_vals, best_times, 'o-', color=c, lw=2.5, markersize=7, label=name)
    axes[1].plot(mb_vals, best_batch_times, 'o-', color=c, lw=2.5, markersize=7, label=name)

for ax, ylabel, title in zip(axes,
        ['Total Training Time (s)', 'Avg Batch Time (s)'],
        ['Total Time (Best Config)', 'Per-Batch Time (Best Config)']):
    ax.set_xlabel('Microbatch Size')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_microbatch.pdf')
plt.show()

## 5. Efficiency Frontier: Loss vs Wall Time

Scatter plot of final val loss vs total wall time for all microbatch sizes, colored by microbatch size.

In [ ]:
n_datasets = len(datasets)
fig, axes = plt.subplots(1, n_datasets, figsize=(6 * n_datasets, 5), squeeze=False)
axes = axes[0]

for col, (name, df) in enumerate(datasets.items()):
    ax = axes[col]
    mb_vals = sorted(df['microbatch_size'].unique())
    n_mb = len(mb_vals)
    mb_colors = [cmap(j / max(n_mb - 1, 1)) for j in range(n_mb)]

    for j, mb in enumerate(mb_vals):
        mb_df = df[df['microbatch_size'] == mb]
        ax.scatter(mb_df['total_time'], mb_df['final_val_loss'],
                   color=mb_colors[j], alpha=0.6, s=40, label=f'mb={mb}')

    ax.set_xlabel('Total Time (s)')
    ax.set_ylabel('Final Val Loss')
    ax.set_yscale('log')
    ax.set_title(name)
    ax.legend(fontsize=8, frameon=False, loc='upper right')

plt.suptitle('Efficiency Frontier: Loss vs Wall Time', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'efficiency_frontier_microbatch.pdf', bbox_inches='tight')
plt.show()

## 6. Heatmaps: k_fraction × Microbatch Size

For each dataset, show the best final val loss (across lr, seeds) as a function of k_fraction and microbatch size.

In [ ]:
n_datasets = len(datasets)
fig, axes = plt.subplots(1, n_datasets, figsize=(5 * n_datasets, 4), squeeze=False)
axes = axes[0]

for col, (name, df) in enumerate(datasets.items()):
    ax = axes[col]

    # Compute k_fraction if not present
    if 'k_fraction' not in df.columns or df['k_fraction'].isna().all():
        if 'k' in df.columns:
            df = df.copy()
            df['k_fraction'] = df['k'] / df['effective_bs']

    pivot = df.pivot_table(
        values='final_val_loss',
        index='k_fraction',
        columns='microbatch_size',
        aggfunc='min'  # best across lr, seed, rtol
    )

    if pivot.empty:
        ax.set_title(f'{name}\n(no data)')
        continue

    im = ax.imshow(np.log10(pivot.values), aspect='auto', cmap='viridis')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([str(int(c)) for c in pivot.columns], rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{kf:.2f}' for kf in pivot.index])
    ax.set_xlabel('Microbatch Size')
    ax.set_ylabel('k / effective_bs')
    ax.set_title(name)

    # Annotate cells
    for yi in range(len(pivot.index)):
        for xi in range(len(pivot.columns)):
            val = pivot.values[yi, xi]
            if not np.isnan(val):
                ax.text(xi, yi, f'{val:.3f}', ha='center', va='center',
                        fontsize=7, color='white' if np.log10(val) < np.log10(pivot.values[~np.isnan(pivot.values)]).mean() else 'black')

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('log$_{10}$(Val Loss)')

plt.suptitle('Best Val Loss: k_fraction × Microbatch Size', y=1.02)
plt.savefig(PLOT_DIR / 'heatmap_kfrac_vs_microbatch.pdf', bbox_inches='tight')
plt.show()

## 7. Singular Value Analysis by Microbatch Size

How does the effective rank of the Jacobian change with microbatching?

In [ ]:
n_datasets = len(datasets)
fig, axes = plt.subplots(1, n_datasets, figsize=(6 * n_datasets, 5), squeeze=False)
axes = axes[0]

for col, (name, df) in enumerate(datasets.items()):
    ax = axes[col]
    bs = df['batch_size'].iloc[0]
    mb_vals = sorted(df['microbatch_size'].unique())
    n_mb = len(mb_vals)
    mb_colors = [cmap(j / max(n_mb - 1, 1)) for j in range(n_mb)]

    for j, mb in enumerate(mb_vals):
        mb_df = df[df['microbatch_size'] == mb]
        best_idx = mb_df['final_val_loss'].idxmin()
        row = df.loc[best_idx]

        svd_info = row.get('svd_info')
        if svd_info is None or not isinstance(svd_info, dict):
            continue
        num_nonzero = svd_info.get('num_nonzero_svs', [])
        if len(num_nonzero) == 0:
            continue

        eff = bs // mb
        n_epoch = len(row['losses']['train'])
        num_nonzero = np.array(num_nonzero)
        # Normalize by effective batch size
        svs_norm = num_nonzero / eff
        x = np.linspace(0, n_epoch, len(svs_norm))

        # Smooth
        sw = max(1, len(svs_norm) // (n_epoch * 3))
        if sw > 1:
            svs_smooth = sliding_average(svs_norm, window=sw)
            ax.plot(x[sw-1:], svs_smooth, color=mb_colors[j], lw=2,
                    label=f'mb={mb} (eff={eff})')
        else:
            ax.plot(x, svs_norm, color=mb_colors[j], lw=2,
                    label=f'mb={mb} (eff={eff})')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Active SVs / Effective BS')
    ax.set_ylim(0, 1.15)
    ax.set_title(name)
    ax.legend(fontsize=8, frameon=False)

plt.suptitle('Effective Rank Evolution by Microbatch Size', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'sv_rank_by_microbatch.pdf', bbox_inches='tight')
plt.show()

## 8. Cross-Dataset Comparison: Normalized Loss Decay

Compare how microbatching degrades convergence across datasets by normalizing
loss to initial value and plotting against training progress.

In [ ]:
colors_ds = sns.color_palette('deep')
linestyles = ['-', '--', ':', '-.', (0, (3, 1, 1, 1)), (0, (5, 2))]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: mb=1 (full per-sample) across datasets
# Right: mb=max across datasets
for ax_idx, mb_selector, title_suffix in [
    (0, 'min', 'mb=1 (full per-sample)'),
    (1, 'max', 'mb=max (near-gradient)'),
]:
    ax = axes[ax_idx]
    for i, (name, df) in enumerate(datasets.items()):
        mb_vals = sorted(df['microbatch_size'].unique())
        mb = mb_vals[0] if mb_selector == 'min' else mb_vals[-1]
        mb_df = df[df['microbatch_size'] == mb]
        if len(mb_df) == 0:
            continue
        best_idx = mb_df['final_val_loss'].idxmin()
        row = df.loc[best_idx]
        val_curve = get_loss_curve(row, 'val')
        val_norm = val_curve / val_curve[0]
        x = np.linspace(0, 1, len(val_norm))
        ax.plot(x, val_norm, color=colors_ds[i % len(colors_ds)], lw=3,
                label=f'{name} (mb={mb})')

    ax.set_xlabel('Training Progress')
    ax.set_ylabel('Val Loss / Initial')
    ax.set_yscale('log')
    ax.set_title(title_suffix)
    ax.legend(frameon=False, fontsize=9)

plt.suptitle('Normalized Loss Decay: Microbatch Extremes', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'normalized_loss_decay_microbatch.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Per-dataset: all microbatch sizes on one plot, normalized loss
n_datasets = len(datasets)
fig, axes = plt.subplots(1, n_datasets, figsize=(6 * n_datasets, 5), squeeze=False)
axes = axes[0]

for col, (name, df) in enumerate(datasets.items()):
    ax = axes[col]
    mb_vals = sorted(df['microbatch_size'].unique())
    n_mb = len(mb_vals)
    mb_colors = [cmap(j / max(n_mb - 1, 1)) for j in range(n_mb)]

    for j, mb in enumerate(mb_vals):
        mb_df = df[df['microbatch_size'] == mb]
        best_idx = mb_df['final_val_loss'].idxmin()
        row = df.loc[best_idx]
        val_curve = get_loss_curve(row, 'val')
        val_norm = val_curve / val_curve[0]
        eff = df['batch_size'].iloc[0] // mb
        ax.plot(range(len(val_norm)), val_norm, color=mb_colors[j], lw=2,
                label=f'mb={mb} (eff={eff})')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val Loss / Initial')
    ax.set_yscale('log')
    ax.set_title(name)
    ax.legend(fontsize=8, frameon=False)

plt.suptitle('Normalized Val Loss by Microbatch Size', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'normalized_loss_all_microbatch.pdf', bbox_inches='tight')
plt.show()

## 9. Summary Table

In [ ]:
# Summary table: best config per dataset per microbatch size
summary_rows = []
for name, df in datasets.items():
    for mb in sorted(df['microbatch_size'].unique()):
        mb_df = df[df['microbatch_size'] == mb]
        best_idx = mb_df['final_val_loss'].idxmin()
        row = df.loc[best_idx]
        eff = df['batch_size'].iloc[0] // mb
        summary_rows.append({
            'Dataset': name,
            'mb': mb,
            'Eff. BS': eff,
            'Best Val Loss': f"{row['final_val_loss']:.4f}",
            'Val Acc': f"{row['final_val_acc']*100:.1f}%" if not np.isnan(row['final_val_acc']) else '—',
            'lr': row['lr'],
            'k': row.get('k', '—'),
            'k_frac': f"{row.get('k_fraction', np.nan):.2f}" if not np.isnan(row.get('k_fraction', np.nan)) else '—',
            'Time (s)': f"{row['total_time']:.1f}",
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))